# telco churn - working out the weights

first pass is raw churn rates. second half is information value, the confound
that nearly caught me, and validating the finished scorecard.

loading

In [26]:
import pandas as pd
df = pd.read_csv('../data/WA_Fn-UseC_-Telco-Customer-Churn.csv')
df.shape
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


check the dtypes. TotalCharges is not a number 

In [27]:
df.dtypes

customerID           object
gender               object
SeniorCitizen         int64
Partner              object
Dependents           object
tenure                int64
PhoneService         object
MultipleLines        object
InternetService      object
OnlineSecurity       object
OnlineBackup         object
DeviceProtection     object
TechSupport          object
StreamingTV          object
StreamingMovies      object
Contract             object
PaperlessBilling     object
PaymentMethod        object
MonthlyCharges      float64
TotalCharges         object
Churn                object
dtype: object

dtype : text , some rows are blank.

In [28]:
bad = pd.to_numeric(df['TotalCharges'],errors='coerce').isna()
bad.sum()
df[bad]

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
488,4472-LVYGI,Female,0,Yes,Yes,0,No,No phone service,DSL,Yes,...,Yes,Yes,Yes,No,Two year,Yes,Bank transfer (automatic),52.55,,No
753,3115-CZMZD,Male,0,No,Yes,0,Yes,No,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,20.25,,No
936,5709-LVOEQ,Female,0,Yes,Yes,0,Yes,No,DSL,Yes,...,Yes,No,Yes,Yes,Two year,No,Mailed check,80.85,,No
1082,4367-NUYAO,Male,0,Yes,Yes,0,Yes,Yes,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,25.75,,No
1340,1371-DWPAZ,Female,0,Yes,Yes,0,No,No phone service,DSL,Yes,...,Yes,Yes,Yes,No,Two year,No,Credit card (automatic),56.05,,No
3331,7644-OMVMY,Male,0,Yes,Yes,0,Yes,No,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,19.85,,No
3826,3213-VVOLG,Male,0,Yes,Yes,0,Yes,Yes,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,25.35,,No
4380,2520-SGTTA,Female,0,Yes,Yes,0,Yes,No,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,20.00,,No
5218,2923-ARZLG,Male,0,Yes,Yes,0,Yes,No,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,One year,Yes,Mailed check,19.70,,No
6670,4075-WKNIU,Female,0,Yes,Yes,0,Yes,Yes,DSL,No,...,Yes,Yes,Yes,No,Two year,No,Mailed check,73.35,,No


11 blanks out of 7043. 

In [29]:
df["tenure"].describe()

count    7043.000000
mean       32.371149
std        24.559481
min         0.000000
25%         9.000000
50%        29.000000
75%        55.000000
max        72.000000
Name: tenure, dtype: float64

every blank row is tenure 0. so they're new
customers who've never been billed. 

In [30]:
df[bad]['tenure']

488     0
753     0
936     0
1082    0
1340    0
3331    0
3826    0
4380    0
5218    0
6670    0
6754    0
Name: tenure, dtype: int64

baseline churn. everything else gets compared to this number.

In [31]:
df['Churn'].value_counts(normalize=True)

Churn
No     0.73463
Yes    0.26537
Name: proportion, dtype: float64

26.5%. 

In [32]:
df.groupby('Contract')['Churn'].value_counts(normalize=True)

Contract        Churn
Month-to-month  No       0.572903
                Yes      0.427097
One year        No       0.887305
                Yes      0.112695
Two year        No       0.971681
                Yes      0.028319
Name: proportion, dtype: float64

churn column, sorted.

In [33]:
df.groupby("Contract")["Churn"].value_counts(normalize=True).unstack()["Yes"].sort_values(ascending=False)

Contract
Month-to-month    0.427097
One year          0.112695
Two year          0.028319
Name: Yes, dtype: float64

42.7% vs 2.8%  huge spread. rest of the columns the same way.

In [34]:
def churn_rate(col):
    return df.groupby(col)["Churn"].value_counts(normalize=True).unstack()["Yes"].sort_values(ascending=False)

for col in ["InternetService", "PaymentMethod", "TechSupport", "OnlineSecurity", "Dependents", "Partner", "SeniorCitizen", "PaperlessBilling", "gender"]:
    print(churn_rate(col), "\n")

InternetService
Fiber optic    0.418928
DSL            0.189591
No             0.074050
Name: Yes, dtype: float64 

PaymentMethod
Electronic check             0.452854
Mailed check                 0.191067
Bank transfer (automatic)    0.167098
Credit card (automatic)      0.152431
Name: Yes, dtype: float64 

TechSupport
No                     0.416355
Yes                    0.151663
No internet service    0.074050
Name: Yes, dtype: float64 

OnlineSecurity
No                     0.417667
Yes                    0.146112
No internet service    0.074050
Name: Yes, dtype: float64 

Dependents
No     0.312791
Yes    0.154502
Name: Yes, dtype: float64 

Partner
No     0.329580
Yes    0.196649
Name: Yes, dtype: float64 

SeniorCitizen
1    0.416813
0    0.236062
Name: Yes, dtype: float64 

PaperlessBilling
Yes    0.335651
No     0.163301
Name: Yes, dtype: float64 

gender
Female    0.269209
Male      0.261603
Name: Yes, dtype: float64 



tenure is continuous so it needs bands. first attempt with bins starting
at 0:

In [35]:
pd.cut(df["tenure"], bins=[0, 6, 12, 24, 48, 100]).isna().sum()


11

11 rows went NaN - `pd.cut` leaves out the left edge, so tenure 0 fell
outside. the same 11 rows as TotalCharges. starting the bins at -1 fixes it.

In [36]:
bands = pd.cut(df["tenure"], bins=[-1, 6, 12, 24, 48, 100],
               labels=["0-6", "7-12", "13-24", "25-48", "49+"])
df.groupby(bands, observed=True)["Churn"].value_counts(normalize=True).unstack()["Yes"]


tenure
0-6      0.529372
7-12     0.358865
13-24    0.287109
25-48    0.203890
49+      0.095132
Name: Yes, dtype: float64

fix in place.

In [37]:
bands = pd.cut(df["tenure"], bins=[-1, 6, 12, 24, 48, 100],
               labels=["0-6", "7-12", "13-24", "25-48", "49+"])
df.groupby(bands, observed=True)["Churn"].value_counts(normalize=True).unstack()["Yes"]

tenure
0-6      0.529372
7-12     0.358865
13-24    0.287109
25-48    0.203890
49+      0.095132
Name: Yes, dtype: float64

52.9% down to 9.5%. sanity check the range.

In [38]:
df["tenure"].describe()

count    7043.000000
mean       32.371149
std        24.559481
min         0.000000
25%         9.000000
50%        29.000000
75%        55.000000
max        72.000000
Name: tenure, dtype: float64

monthly charges 

In [39]:
df["MonthlyCharges"].describe()

count    7043.000000
mean       64.761692
std        30.090047
min        18.250000
25%        35.500000
50%        70.350000
75%        89.850000
max       118.750000
Name: MonthlyCharges, dtype: float64

banded.

In [40]:
charge_bands = pd.cut(df["MonthlyCharges"], bins=[0, 35, 60, 85, 200],
                      labels=["<35", "35-60", "60-85", "85+"])
df.groupby(charge_bands, observed=True)["Churn"].value_counts(normalize=True).unstack()["Yes"]

MonthlyCharges
<35      0.108934
35-60    0.254438
60-85    0.329892
85+      0.338208
Name: Yes, dtype: float64

remaining columns

In [41]:
for col in ["OnlineBackup", "DeviceProtection", "StreamingTV", "StreamingMovies", "MultipleLines", "PhoneService"]:
    print(churn_rate(col), "\n")

OnlineBackup
No                     0.399288
Yes                    0.215315
No internet service    0.074050
Name: Yes, dtype: float64 

DeviceProtection
No                     0.391276
Yes                    0.225021
No internet service    0.074050
Name: Yes, dtype: float64 

StreamingTV
No                     0.335231
Yes                    0.300702
No internet service    0.074050
Name: Yes, dtype: float64 

StreamingMovies
No                     0.336804
Yes                    0.299414
No internet service    0.074050
Name: Yes, dtype: float64 

MultipleLines
Yes                 0.286099
No                  0.250442
No phone service    0.249267
Name: Yes, dtype: float64 

PhoneService
Yes    0.267096
No     0.249267
Name: Yes, dtype: float64 



## weight of evidence / information value

the rates above tell me which groups are risky but not how much each COLUMN is
worth next to the others. is contract twice payment method, or ten times?


information value is the standard measure for this. 

    < 0.02   useless
    0.02-0.1 weak
    0.1-0.3  medium
    0.3-0.5  strong
    > 0.5    suspicious - go find the confound

In [42]:
import numpy as np

def woe_iv(col, data=None):
    """WoE per group and total IV for one column."""
    d = df if data is None else data
    t = pd.crosstab(d[col], d["Churn"])
    good = t["No"] / t["No"].sum()      # share of non-churners
    bad = t["Yes"] / t["Yes"].sum()     # share of churners
    woe = np.log(bad / good)
    return pd.DataFrame({
        "n": t.sum(axis=1),
        "churn_rate": (t["Yes"] / t.sum(axis=1)).round(3),
        "woe": woe.round(3),
        "iv": ((bad - good) * woe).round(4),
    })

def iv(col, data=None):
    return woe_iv(col, data)["iv"].sum()

woe_iv("Contract")

,n,churn_rate,woe,iv
Contract,,,,
Month-to-month,3875,0.427,0.725,0.3307
One year,1473,0.113,-1.045,0.1712
Two year,1695,0.028,-2.517,0.7367


tenure needs the same bands the scorer uses.

In [43]:
# tenure is continuous, so IV needs the same bands the scorer uses
tenure_banded = df.assign(tenure_band=pd.cut(
    df["tenure"], bins=[-1, 6, 12, 24, 48, 100],
    labels=["0-6", "7-12", "13-24", "25-48", "49+"]))

woe_iv("tenure_band", tenure_banded)

,n,churn_rate,woe,iv
tenure_band,,,,
0-6,1481,0.529,1.136,0.3235
7-12,705,0.359,0.438,0.0210
13-24,1024,0.287,0.109,0.0018
25-48,1594,0.204,-0.344,0.0245
49+,2239,0.095,-1.234,0.3426


ranked column

In [44]:
# every candidate column, ranked
rows = {}
for c in ["Contract", "InternetService", "PaymentMethod", "OnlineSecurity",
          "TechSupport", "OnlineBackup", "DeviceProtection", "StreamingTV",
          "StreamingMovies", "PaperlessBilling", "Dependents", "Partner",
          "SeniorCitizen", "MultipleLines", "PhoneService", "gender"]:
    rows[c] = iv(c)
rows["tenure_band"] = iv("tenure_band", tenure_banded)

pd.Series(rows).sort_values(ascending=False).round(3)

Contract            1.239
OnlineSecurity      0.718
tenure_band         0.713
TechSupport         0.700
InternetService     0.618
OnlineBackup        0.529
DeviceProtection    0.500
PaymentMethod       0.457
StreamingMovies     0.381
StreamingTV         0.380
PaperlessBilling    0.203
Dependents          0.156
Partner             0.119
SeniorCitizen       0.106
MultipleLines       0.008
PhoneService        0.001
gender              0.000
dtype: float64

six columns over 0.3 and one over 0.7. that's more strong predictors than a
dataset this size should have, and >0.5 literally means go and check.

every service column has a `No internet service` value in it.

In [45]:
churn_rate("InternetService")

InternetService
Fiber optic    0.418928
DSL            0.189591
No             0.074050
Name: Yes, dtype: float64

7.4% against a 26.5% baseline - the most loyal segment in the data.

so the add-on columns aren't measuring the add-on. a third of the rows in
OnlineSecurity just say "no internet", and that one fact does the predicting.
every add-on column is inheriting InternetService's power and reporting it as
its own.

recompute on internet-having customers only, where yes/no means yes/no:

In [46]:
internet = df[df["InternetService"] != "No"]
print("internet-having customers:", len(internet), "of", len(df))

confound = pd.DataFrame({
    "raw_iv": {c: iv(c) for c in ["OnlineSecurity", "TechSupport", "OnlineBackup",
                                  "DeviceProtection", "StreamingTV", "StreamingMovies"]},
    "true_iv": {c: iv(c, internet) for c in ["OnlineSecurity", "TechSupport", "OnlineBackup",
                                             "DeviceProtection", "StreamingTV", "StreamingMovies"]},
})
confound["change_%"] = ((confound["true_iv"] / confound["raw_iv"] - 1) * 100).round(0)
confound.round(3).sort_values("true_iv", ascending=False)

internet-having customers: 5517 of 7043


,raw_iv,true_iv,change_%
OnlineSecurity,0.718,0.416,-42.0
TechSupport,0.700,0.394,-44.0
OnlineBackup,0.529,0.185,-65.0
DeviceProtection,0.500,0.150,-70.0
StreamingMovies,0.381,0.007,-98.0
StreamingTV,0.380,0.006,-98.0


**streaming goes 0.38 -> 0.006. a 98% drop.** it read as strong and is worth
nothing. weighting off the raw number would have put real points on a column
that predicts nothing.

support and security hold at ~0.40, backup and protection drop to ~0.15. so the
four add-ons aren't equal - that's the 4/4/2/2 split instead of 3/3/3/3.

it also decides the code: three-way, not two-way. a phone-only customer
"missing" tech support isn't at risk, they never had internet. collapsing that
to `No` would put points on the most loyal segment. 

## dropping

knowing what doesn't predict is half the job.

In [47]:
pd.Series({c: iv(c) for c in ["gender", "PhoneService", "MultipleLines",
                              "StreamingTV", "StreamingMovies"]}).round(4)

gender             0.0004
PhoneService       0.0008
MultipleLines      0.0083
StreamingTV        0.3804
StreamingMovies    0.3814
dtype: float64

TotalCharges , it's redundant - check it against tenure x
monthly.

In [48]:
# TotalCharges is not weak - it is redundant. it restates tenure x monthly.
total = pd.to_numeric(df["TotalCharges"], errors="coerce")
implied = df["tenure"] * df["MonthlyCharges"]
ok = total.notna()

print("corr(TotalCharges, tenure x MonthlyCharges) = %.4f" % total[ok].corr(implied[ok]))

corr(TotalCharges, tenure x MonthlyCharges) = 0.9996


gender 0.0004 - nothing. wouldn't have used it anyway, picking who gets a
retention offer by gender is a fairness problem before it's a stats one.

phone service and multiple lines, nothing.

TotalCharges 0.9996 correlated with tenure x monthly. not a second signal,
it's tenure wearing a hat.

## does it actually work

weights are set now. question is whether the score separates people who really
churned.

caveat: measuring on the data i tuned on, so this tests the RANKING, not
out-of-sample accuracy.

importing the real scorer from the backend rather than rewriting the rules here
- otherwise this could pass while the api does something else.

In [49]:
import sys
sys.path.insert(0, "../backend")

from app.data_access.loader import load_customers
from app.services.scoring import score_customer

customers = load_customers()
assessed = {cid: score_customer(c) for cid, c in customers.items()}

df["score"] = df["customerID"].map(lambda c: assessed[c].score)
df["tier"] = df["customerID"].map(lambda c: assessed[c].tier)

df[["customerID", "tenure", "Contract", "score", "tier", "Churn"]].head()

,customerID,tenure,Contract,score,tier,Churn
0,7590-VHVEG,1,Month-to-month,87,CRITICAL,No
1,5575-GNVDE,34,One year,33,MEDIUM,No
2,3668-QPYBK,2,Month-to-month,76,CRITICAL,Yes
3,7795-CFOCW,45,One year,25,MEDIUM,No
4,9237-HQITU,2,Month-to-month,98,CRITICAL,Yes


band separation - is each tier a usable worklist?

In [50]:
# band separation - is each tier a usable worklist?
tier_order = ["LOW", "MEDIUM", "HIGH", "CRITICAL"]
df.groupby("tier").agg(
    customers=("score", "size"),
    churn_rate=("Churn", lambda s: (s == "Yes").mean()),
).reindex(tier_order).round(3)

,customers,churn_rate
tier,,
LOW,1867,0.028
MEDIUM,1428,0.116
HIGH,1978,0.287
CRITICAL,1770,0.613


and deciles, to check the ranking holds all the way down and not just at
the edges.

In [51]:
# deciles - does the ranking hold all the way down, or only at the edges?
decile = pd.qcut(df["score"].rank(method="first"), 10, labels=False)
by_decile = df.groupby(decile)["Churn"].apply(lambda s: (s == "Yes").mean())

print(by_decile.round(3).to_string())
print("\nmonotonic:", by_decile.is_monotonic_increasing)

score
0    0.011
1    0.023
2    0.060
3    0.108
4    0.155
5    0.232
6    0.314
7    0.463
8    0.580
9    0.709

monotonic: True


---

## what i'm taking from this

- baseline **26.5%**, everything sized off distance from that
- **contract strongest** (iv 1.24), and the only factor an agent can change on
  a call - you can offer a twelve-month contract, you can't offer more tenure.
  most points.
- **tenure second (0.71) but capped below contract** - they overlap,
  month-to-month customers are disproportionately new. two correlated signals
  shouldn't own 60% of the score.
- **streaming dropped** - 0.38 raw, 0.006 real
- **add-ons 4/4/2/2**, not equal
- **gender / phone / multiple lines: no signal. TotalCharges redundant.** all
  dropped, and /model/info says so with the reason
